In [ ]:
import pandas as pd
import numpy as np

column_types = {
    "id": "Int64",
    "osm_id": "str",
    "addr_street": "str",
    "addr_number": "str",
    "ground_floor_group": "str",
    "ground_floor_function": "str",
    "upper_floor_group": "str",
    "upper_floor_function": "str",
    "number_of_storeys": "Int64",
    "is_mixed_use": "boolean",
    "build_year_est": "Int64",
    "arch_style": "str",
    "condition": "Int64",
    "max_height": "float64",
    "is_heritage": "str",
    "source": "str",
    "confidence": "Int64",
    "date_of_checking": "str",
    "number_of_functions": "Int64",
}

In [ ]:
data_gdynia = pd.read_csv(
    "../../data/raw/srodmiescie_raw_object_id.csv", sep=";", index_col=0, dtype=column_types
).dropna(subset="ground_floor_function")

In [ ]:
data_gdynia["date_of_checking"] = pd.to_datetime(
    data_gdynia["date_of_checking"], format="%d.%m.%Y"
)

In [ ]:
data_clean = data_gdynia.assign(
    ground_floor_function=lambda d: (
        d["ground_floor_function"].str.replace("_", " ").str.lower()
    )
)

In [ ]:
street_fixes = {
    "Hryniewickiego": "Antoniego Hryniewickiego",
    "Wendy": "Tadeusza Wendy",
    "Plac Kaszubski - Jana z Kolna": "Jana z Kolna",
}

data_clean["addr_street"] = data_clean["addr_street"].replace(street_fixes)

data_clean["addr_number"] = data_clean["addr_number"].str.strip().str.upper()

In [ ]:
style_fixes = {
    "nowożytnośc": "współczesność",
    "współczsnośc": "współczesność", 
    "wspóółczesność": "współczesność",
    "wspóczesność": "współczesność",
    "współczesnosc": "współczesność",
    "traycjonalizm": "tradycjonalizm",
    "wiejski": "tradycjonalizm",
    "art. deco": "modernizm",
    "artdeco": "modernizm",
    "wertykalizm": "modernizm",
    "nowożytność" : "współczesność",
}

data_clean["arch_style"] = data_clean["arch_style"].str.lower().str.strip().replace(style_fixes)

In [ ]:
data_long = (
    data_clean.assign(
        ground_floor_function=lambda d: d["ground_floor_function"].str.split(",")
    )
    .explode("ground_floor_function")
    .assign(ground_floor_function=lambda d: d["ground_floor_function"].str.strip())
)
data_long = data_long.loc[:, ["addr_street", "ground_floor_function"]]


In [ ]:
num_func_buil = data_long.groupby(level=0).agg(
    number_of_functions=("ground_floor_function", "count")
)

data_clean = pd.merge(
    data_clean, num_func_buil, left_index=True, right_index=True, how="outer"
)

In [ ]:
data_clean.to_csv(
    "../../data/processed/fixed/srodmiescie.csv", sep=";", encoding="utf-8"
)
data_long.to_csv(
    "../../data/processed/fixed/srodmiescie_long.csv", sep=";", encoding="utf-8"
)

In [ ]:
data_clean

In [ ]:
category_map = pd.read_csv(
    "../../data/reference/category_map.csv",
    sep=";",
    index_col=0,
    dtype=column_types,
)

In [ ]:
data_long_categories = pd.merge(
    data_long,
    category_map,
    left_on="ground_floor_function",
    right_index=True,
    indicator=True,
    how="outer",
    validate="many_to_one",
)

In [ ]:
data_long_export = data_long_categories.loc[:,['addr_street','function_clean','category']]
data_long_export.to_csv('../../data/processed/fixed/srodmiescie_long_categories.csv', sep=";", encoding="utf-8")